In [ ]:
import pandas as pd

# Memuat dataset
df = pd.read_csv('/content/reviews.csv')

# 1. Menampilkan jumlah data yang kosong per kolom
print("Jumlah data kosong per kolom:")
print(df.isnull().sum())

# 2. Menampilkan total SELURUH sel yang kosong di dalam tabel
print("\nTotal seluruh sel yang kosong:", df.isnull().sum().sum())

# 3. Menampilkan total semua baris dan kolom
print("\nTotal (baris, kolom):", df.shape)
print("Total keseluruhan baris:", len(df))

Jumlah data kosong per kolom:
reviews    0
label      0
dtype: int64

Total seluruh sel yang kosong: 0

Total (baris, kolom): (831, 2)
Total keseluruhan baris: 831


In [4]:
!pip install datasets pandas matplotlib seaborn scikit-learn Sastrawi nltk


In [10]:
import pandas as pd
import numpy as np
import re
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

# Text Preprocessing dari Sastrawi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Menetapkan random seed untuk reproducibility hasil
import random
random.seed(42)
np.random.seed(42)

print("Memulai Tahap Data Preparation (Bab 5)...\n")

# ==============================================================================
# 0. MEMUAT DATASET (MELANJUTKAN DARI BAB 4)
# ==============================================================================
print("1. Memuat dataset dari Hugging Face...")
ds = load_dataset("AIbnuHibban/e-commerce-sentiment-bahasa-indonesia")
df = pd.DataFrame(ds['train'])

df = pd.DataFrame(df)
print(f"Dataset dimuat. Jumlah baris awal: {df.shape[0]}")


# ==============================================================================
# 5.1 DATA CLEANING & FEATURE EXTRACTION (PSEUDO-LABELING)
# ==============================================================================
print("\n2. Melakukan Data Cleaning & Extract Aspek...")

# A. Menghapus Duplikat (Menghindari Data Leakage)
# Catatan: EDA menemukan >67% duplikat teks di dataset asli.
df_bersih = df.drop_duplicates(subset=['comment']).copy()
print(f"-> Setelah hapus duplikat: {len(df_bersih)} baris tersisa.")

# B. Menghapus kolom 'category' bawaan
if 'category' in df_bersih.columns:
    df_bersih = df_bersih.drop(columns=['category'])
    print("-> Kolom 'category' bawaan berhasil dihapus.")

# C. Menangani Noise (Menghapus komentar terlalu pendek)
# Catatan: EDA menemukan ulasan dengan panjang < 5 karakter.
df_bersih['char_length'] = df_bersih['comment'].astype(str).str.len()
df_bersih = df_bersih[df_bersih['char_length'] >= 5]
df_bersih = df_bersih.drop(columns=['char_length']) # Hapus kolom bantuan
print(f"-> Setelah hapus ulasan < 5 karakter: {len(df_bersih)} baris tersisa.")


# D. Aspect Pseudo-Labeling (Rule-Based untuk 3 Aspek Utama)
def tentukan_aspek(row):
    teks = str(row['comment']).lower()

    kw_pengiriman = ['kurir', 'kirim', 'paket', 'packing', 'ongkir', 'datang', 'sampai', 'lama', 'cepat', 'ekspedisi', 'kurirnya']
    kw_layanan = ['admin', 'seller', 'respon', 'balas', 'chat', 'pelayanan', 'toko', 'cs', 'penjual']

    if any(word in teks for word in kw_pengiriman):
        return 'Pengiriman'
    elif any(word in teks for word in kw_layanan):
        return 'Layanan Pelanggan'
    else:
        # Jika tidak ada kata kunci pengiriman/layanan, berarti membahas barang
        return 'Produk'

df_bersih['aspek'] = df_bersih.apply(tentukan_aspek, axis=1)
print("\nDistribusi 3 Aspek Utama:")
print(df_bersih['aspek'].value_counts())


# ==============================================================================
# 5.3 DATA TRANSFORMATION (TEXT PREPROCESSING & ENCODING)
# ==============================================================================
print("\n3. Melakukan Text Preprocessing & Label Encoding...")

# Inisiasi Sastrawi (Bisa memakan waktu komputasi yang lama pada data riil)
# factory_stop = StopWordRemoverFactory()
# stopword_remover = factory_stop.create_stop_word_remover()
# factory_stem = StemmerFactory()
# stemmer = factory_stem.create_stemmer()

def bersihkan_teks(teks):
    # 1. Lowercasing
    teks = str(teks).lower()
    # 2. Cleansing: Hapus angka, tanda baca, emoji (hanya simpan huruf abjad a-z dan spasi)
    teks = re.sub(r'[^a-z\s]', '', teks)

    # 3. Stopwords & Stemming (Uncomment jika menggunakan Sastrawi)
    # teks = stopword_remover.remove(teks)
    # teks = stemmer.stem(teks)

    # 4. Hapus spasi berlebih
    teks = re.sub(r'\s+', ' ', teks).strip()
    return teks

df_bersih['comment_clean'] = df_bersih['comment'].apply(bersihkan_teks)

# Hapus baris yang mungkin menjadi kosong (string kosong) setelah proses pembersihan
df_bersih = df_bersih[df_bersih['comment_clean'] != '']

# Label Encoding untuk Sentimen (Variabel Target y)
le_sentiment = LabelEncoder()
df_bersih['sentiment_encoded'] = le_sentiment.fit_transform(df_bersih['sentiment'])
print("\nMapping Label Sentimen ke Angka:")
for i, label in enumerate(le_sentiment.classes_):
    print(f"  {label} -> {i}")

# Label Encoding untuk Aspek (Opsional: Digunakan jika memprediksi 2 target)
le_aspek = LabelEncoder()
df_bersih['aspek_encoded'] = le_aspek.fit_transform(df_bersih['aspek'])


# ==============================================================================
# 5.4 DATA SPLITTING (STRATIFIED)
# ==============================================================================
print("\n4. Melakukan Data Splitting...")

X = df_bersih['comment_clean']
y = df_bersih['sentiment_encoded']

# Catatan: Stratified Split sangat penting untuk menjaga porsi kelas 'Netral' yang sedikit.
# Menggunakan rasio 80% (Train), 10% (Validation), 10% (Test).

try:
    # 1. Pisahkan Data Test (10% dari total awal)
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=0.10, random_state=42, stratify=y
    )

    # 2. Pisahkan Data Train (80% dari total awal) dan Val (10% dari total awal) dari X_temp
    # Proporsi 10% dari total awal setara dengan ~11.11% dari X_temp (10/90)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.1111, random_state=42, stratify=y_temp
    )

    print("\nDistribusi Himpunan Data (Rasio 80:10:10):")
    print(f"  - Data Training   : {len(X_train)} sampel")
    print(f"  - Data Validation : {len(X_val)} sampel")
    print(f"  - Data Testing    : {len(X_test)} sampel")

except ValueError as e:
    # Blok try-except ini ditambahkan agar script tidak error saat dicoba dengan data simulasi yang sangat kecil
    print("\n[Peringatan] Dataset simulasi terlalu kecil untuk stratified split. Melakukan split standar 80:20...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
    X_val, y_val = X_test, y_test # Dummy val set untuk testing
    print(f"  - Data Training : {len(X_train)} sampel")
    print(f"  - Data Testing  : {len(X_test)} sampel")


# ==============================================================================
# 5.2 FEATURE ENGINEERING (TF-IDF)
# ==============================================================================
print("\n5. Melakukan Ekstraksi Fitur Teks (TF-IDF)...")

# Inisialisasi parameter TF-IDF (Membatasi pada 5.000 bobot fitur)
tfidf_vectorizer = TfidfVectorizer(max_features=5000)

# MENCEGAH DATA LEAKAGE:
# Object TfidfVectorizer HARUS DI-FIT HANYA PADA X_train.
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

# Untuk validation dan test, kita HANYA melakukan Transform.
X_val_tfidf = tfidf_vectorizer.transform(X_val)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f"Dimensi Matriks Fitur TF-IDF (Train): {X_train_tfidf.shape}")

print("\n✅ SELESAI! Data Preparation berhasil dieksekusi tanpa data leakage.")
print("Variabel (X_train_tfidf, y_train) siap diumpankan ke Model Baseline (Logistic Regression) dan Model ML (LightGBM).")

Memulai Tahap Data Preparation (Bab 5)...

1. Memuat dataset dari Hugging Face...
Dataset dimuat. Jumlah baris awal: 21840

2. Melakukan Data Cleaning & Extract Aspek...
-> Setelah hapus duplikat: 7025 baris tersisa.
-> Kolom 'category' bawaan berhasil dihapus.
-> Setelah hapus ulasan < 5 karakter: 7018 baris tersisa.

Distribusi 3 Aspek Utama:
aspek
Produk               5737
Layanan Pelanggan    1092
Pengiriman            189
Name: count, dtype: int64

3. Melakukan Text Preprocessing & Label Encoding...

Mapping Label Sentimen ke Angka:
  negative -> 0
  neutral -> 1
  positive -> 2

4. Melakukan Data Splitting...

Distribusi Himpunan Data (Rasio 80:10:10):
  - Data Training   : 5614 sampel
  - Data Validation : 702 sampel
  - Data Testing    : 702 sampel

5. Melakukan Ekstraksi Fitur Teks (TF-IDF)...
Dimensi Matriks Fitur TF-IDF (Train): (5614, 989)

✅ SELESAI! Data Preparation berhasil dieksekusi tanpa data leakage.
Variabel (X_train_tfidf, y_train) siap diumpankan ke Model Baseline

In [12]:
print(df.isnull().sum())
df_kategori_kosong = df[df['category'].isnull()]

print(f"Total baris dengan kategori kosong: {len(df_kategori_kosong)}")
print("\nMenampilkan 10 contoh ulasan dengan kategori kosong:")

# Menampilkan 10 baris pertama
display(df_kategori_kosong[['comment', 'sentiment', 'category', 'rating']].head(10))

comment          0
sentiment        0
category     17000
rating           0
dtype: int64
Total baris dengan kategori kosong: 17000

Menampilkan 10 contoh ulasan dengan kategori kosong:


,comment,sentiment,category,rating
4840,"Wow, senang banget ditipu!",negative,None,1
4841,Shock! Ga nyangka bakal sekeren ini produknya.,positive,None,5
4842,"Top reviewnya, kecewa barangnya!",negative,None,1
4843,"Bukan jelek kok, malah oke!",positive,None,5
4844,"Setan nih seller, setan baiknya kasih kualitas...",positive,None,5
4845,"Agak bagus, agak kurang juga.",neutral,None,3
4846,Produk premium dengan harga yang masuk akal. R...,positive,None,5
4847,"Kenapa bisa gini sih, mengecewakan",negative,None,2
4848,"Gimana nih dong, kecewa parah",negative,None,2
4849,Sesuai harga,neutral,None,3
